In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

USE_LOG = True
CORR_METHOD = "pearson"       # "pearson" 或 "spearman"
CLIP_QUANTILE = (0.01, 0.99)   # 不裁剪设为 None

FILTER_EXPRESSED = True
FILTER_MODE = "both"           # "both": RNA和ADT都大于阈值; "either": 任一大于阈值
MIN_RNA_VALUE = 0.025            
MIN_ADT_VALUE = 0.025

OUTDIR = Path("./metacell_rna_adt_marker_scatter")
OUTDIR.mkdir(parents=True, exist_ok=True)

algo_file_map = {
    "Supercell": {
        "rna_h5ad": "/workspace/garq/results/5_supercell/three/GSE158013/GSE158013_rna_metacells.h5ad",
        "adt_h5ad": "/workspace/garq/results/5_supercell/three/GSE158013/GSE158013_adt_metacells.h5ad",
    },
    "SEACell": {
        "rna_h5ad": "/workspace/garq/results/3_seacell/three/GSE158013/GSE158013_rna_metacells.h5ad",
        "adt_h5ad": "/workspace/garq/results/3_seacell/three/GSE158013/GSE158013_adt_metacells.h5ad",
    },
    "MetaCellV2": {
        "rna_h5ad": "/workspace/garq/results/4_mc2/three/GSE158013/GSE158013_rna_metacells.h5ad",
        "adt_h5ad": "/workspace/garq/results/4_mc2/three/GSE158013/GSE158013_adt_metacells.h5ad",
    },
}

marker_pairs = [
    {"gene": "CD8A", "protein": "CD8a"},
    {"gene": "NCAM1", "protein": "CD56"},
]

REF_RNA_H5AD = None
REF_ADT_H5AD = None

CELLTYPE_CANDIDATES = ["celltype", "cell_type", "major_cell_type", "leiden", "cluster"]

color_map = {
    "T.CD4.Naive": "#FF6B6B",
    "T.CD4.Memory": "#FF9999",
    "T.CD8.Naive": "#4ECDC4",
    "T.CD8.Effector": "#2C7A7B",
    "T.DoubleNegative": "#A8DADC",
    "B.Naive": "#FFD93D",
    "B.Activated": "#FFA500",
    "Mono.CD14": "#9D4EDD",
    "Mono.CD16": "#C77DFF",
    "NK": "#06D6A0",
    "DC.Myeloid": "#17becf",
    "Platelets": "#95A5A6",
}


def find_feature_name(adata, feature_name):
    if feature_name in adata.var_names:
        return feature_name

    lower_map = {str(v).lower(): str(v) for v in adata.var_names}
    key = feature_name.lower()

    if key in lower_map:
        return lower_map[key]

    raise KeyError(
        f"找不到特征: {feature_name}\n"
        f"var_names 前20个: {adata.var_names[:20].tolist()}"
    )


def get_vector(adata, feature_name):
    feature_name = find_feature_name(adata, feature_name)
    x = adata[:, feature_name].X

    if hasattr(x, "toarray"):
        x = x.toarray().flatten()
    else:
        x = np.asarray(x).flatten()

    return x.astype(float), feature_name


def maybe_restore_var_names(rna, adt):
    if REF_RNA_H5AD is not None:
        ref_rna = sc.read_h5ad(REF_RNA_H5AD)
        if rna.n_vars == ref_rna.n_vars:
            rna.var_names = ref_rna.var_names

    if REF_ADT_H5AD is not None:
        ref_adt = sc.read_h5ad(REF_ADT_H5AD)
        if adt.n_vars == ref_adt.n_vars:
            adt.var_names = ref_adt.var_names

    return rna, adt


def get_cell_types(adata):
    for col in CELLTYPE_CANDIDATES:
        if col in adata.obs.columns:
            return adata.obs[col].astype(str).values, col
    return None, None


def add_unknown_colors(cell_types):
    if cell_types is None:
        return

    for ct in np.unique(cell_types):
        if ct not in color_map:
            rng = np.random.default_rng(abs(hash(ct)) % (2**32))
            color_map[ct] = "#{:02x}{:02x}{:02x}".format(
                rng.integers(0, 256),
                rng.integers(0, 256),
                rng.integers(0, 256),
            )


def transform_values(values):
    values = np.asarray(values, dtype=float)

    if USE_LOG:
        values = np.log1p(np.maximum(values, 0))
        transform_label = "log1p"
    else:
        transform_label = "raw"

    return values, transform_label


def prepare_pair_values(x_raw, y_raw):
    if len(x_raw) != len(y_raw):
        raise ValueError(f"RNA 和 ADT metacell 数量不一致: RNA={len(x_raw)}, ADT={len(y_raw)}")

    x_raw = np.asarray(x_raw, dtype=float)
    y_raw = np.asarray(y_raw, dtype=float)

    valid_mask = np.isfinite(x_raw) & np.isfinite(y_raw)
    kept_mask = valid_mask.copy()

    if FILTER_EXPRESSED:
        if FILTER_MODE == "both":
            expressed_mask = (x_raw > MIN_RNA_VALUE) & (y_raw > MIN_ADT_VALUE)
        elif FILTER_MODE == "either":
            expressed_mask = (x_raw > MIN_RNA_VALUE) | (y_raw > MIN_ADT_VALUE)
        else:
            raise ValueError(f"未知 FILTER_MODE: {FILTER_MODE}")

        kept_mask = valid_mask & expressed_mask

    print(
        f"Filter kept {kept_mask.sum()} / {len(kept_mask)} metacells "
        f"(mode={FILTER_MODE}, RNA>{MIN_RNA_VALUE}, ADT>{MIN_ADT_VALUE})"
    )

    x = x_raw[kept_mask]
    y = y_raw[kept_mask]

    x, transform_label = transform_values(x)
    y, _ = transform_values(y)

    if CLIP_QUANTILE is not None and len(x) > 0:
        q_low, q_high = CLIP_QUANTILE

        x_low, x_high = np.quantile(x, [q_low, q_high])
        y_low, y_high = np.quantile(y, [q_low, q_high])

        x = np.clip(x, x_low, x_high)
        y = np.clip(y, y_low, y_high)

    return x, y, kept_mask, transform_label


def compute_correlation(x, y):
    if len(x) < 3:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    if CORR_METHOD == "pearson":
        return pearsonr(x, y)
    elif CORR_METHOD == "spearman":
        return spearmanr(x, y)

    raise ValueError(f"未知相关性方法: {CORR_METHOD}")


def format_pvalue(p):
    if np.isnan(p):
        return "NA"
    if p < 1e-300:
        return "<1e-300"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.2f}"


def plot_one_pair_for_all_algorithms(pair):
    gene_name = pair["gene"]
    protein_name = pair["protein"]

    n_algos = len(algo_file_map)
    fig, axes = plt.subplots(1, n_algos, figsize=(3 * n_algos, 3.35), squeeze=False)
    axes = axes[0]

    stats_rows = []

    for ax, (algo_name, paths) in zip(axes, algo_file_map.items()):
        print("=" * 80)
        print(f"Processing: {algo_name} | {gene_name} - {protein_name}")
        print("=" * 80)

        rna = sc.read_h5ad(paths["rna_h5ad"])
        adt = sc.read_h5ad(paths["adt_h5ad"])
        rna, adt = maybe_restore_var_names(rna, adt)

        rna_gene_raw, gene_used = get_vector(rna, gene_name)
        adt_protein_raw, protein_used = get_vector(adt, protein_name)

        rna_plot, adt_plot, kept_mask, transform_label = prepare_pair_values(
            rna_gene_raw,
            adt_protein_raw,
        )

        rho, p_value = compute_correlation(rna_plot, adt_plot)
        p_text = format_pvalue(p_value)

        cell_types, celltype_col = get_cell_types(rna)
        if cell_types is not None and len(cell_types) == len(kept_mask):
            cell_types = cell_types[kept_mask]
        else:
            cell_types = None

        add_unknown_colors(cell_types)

        if cell_types is not None and len(cell_types) == len(rna_plot):
            for ct in np.unique(cell_types):
                mask = cell_types == ct
                ax.scatter(
                    rna_plot[mask],
                    adt_plot[mask],
                    s=20,
                    alpha=0.55,
                    color=color_map.get(ct, "#999999"),
                    edgecolors="none",
                    rasterized=False,
                )
        else:
            ax.scatter(
                rna_plot,
                adt_plot,
                s=20,
                alpha=0.55,
                color="#3C5488",
                edgecolors="none",
                rasterized=False,
            )

        ax.set_xlabel(f"{gene_used} RNA ({transform_label})", fontsize=11, fontstyle="italic")
        ax.set_ylabel(f"{protein_used} ADT ({transform_label})", fontsize=11, fontstyle="italic")
        ax.set_title(f"{algo_name}\ncor = {rho:.2f}, p = {p_text}", fontsize=10, pad=10)
        ax.tick_params(labelsize=9)

        sns.despine(ax=ax)
        for spine in ax.spines.values():
            spine.set_linewidth(1.0)

        stats_rows.append({
            "algorithm": algo_name,
            "gene": gene_used,
            "protein": protein_used,
            "n_metacells_before_filter": len(rna_gene_raw),
            "n_metacells_after_filter": len(rna_plot),
            "correlation_method": CORR_METHOD,
            "correlation_r": rho,
            "p_value": p_value,
            "p_value_formatted": p_text,
            "use_log": USE_LOG,
            "transform": transform_label,
            "clip_quantile": str(CLIP_QUANTILE),
            "filter_expressed": FILTER_EXPRESSED,
            "filter_mode": FILTER_MODE,
            "min_rna_value": MIN_RNA_VALUE,
            "min_adt_value": MIN_ADT_VALUE,
            "celltype_col": celltype_col,
            "rna_h5ad": paths["rna_h5ad"],
            "adt_h5ad": paths["adt_h5ad"],
        })

        print(f"{algo_name}: n={len(rna_plot)}, cor={rho:.4f}, p={p_text}")

    plt.tight_layout()

    base_name = f"{gene_name}_{protein_name}_metacell_3algorithms_{CORR_METHOD}_rawfilter"
    pdf_path = OUTDIR / f"{base_name}.pdf"
    png_path = OUTDIR / f"{base_name}.png"

    plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
    plt.savefig(png_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved: {pdf_path}")
    print(f"Saved: {png_path}")

    return stats_rows


all_stats = []

for pair in marker_pairs:
    all_stats.extend(plot_one_pair_for_all_algorithms(pair))

stats_df = pd.DataFrame(all_stats)
stats_path = OUTDIR / "metacell_rna_adt_marker_correlations_with_pvalue.csv"
stats_df.to_csv(stats_path, index=False)

print("\nCorrelation summary:")
print(stats_df)
print(f"\nSaved stats: {stats_path}")
print("\nDone.")